# Búsqueda de hiperparámetros — HVFHS Driver Pay

Experimento de MLflow del TP final de MLOps1 (CEIA-FIUBA).

El modelo predice el **pago al conductor** (`driver_pay`) de viajes HVFHS de NYC,
en escala logarítmica (`driver_pay_log`). Para volver a dólares hay que aplicar `np.exp`.

El ETL ya corrió en Airflow (`process_etl_hvfhs_data`) y dejó train/test escalados en MinIO.
Esta notebook lee esos datos, busca hiperparámetros de **XGBoost**, loguea cada corrida
en MLflow y registra el mejor modelo en el Model Registry.

## Contexto que ya tenemos

### Datos en MinIO (bucket `data`)

| Archivo | Rol |
|---------|-----|
| `s3://data/final/train/hvfhs_X_train.parquet` | Features de train (ya con `StandardScaler`) |
| `s3://data/final/train/hvfhs_y_train.parquet` | Target de train (`driver_pay_log`) |
| `s3://data/final/test/hvfhs_X_test.parquet` | Features de test (escaladas con el scaler de train) |
| `s3://data/final/test/hvfhs_y_test.parquet` | Target de test |
| `s3://data/data_info/data.json` | Media, desvío y nombres de columnas del scaler |

Tamaños del ETL: **train ≈ 399 817** filas, **test ≈ 99 955** filas.

### Infra (desde esta notebook, en tu máquina)

| Servicio | URL |
|----------|-----|
| MLflow | `http://localhost:5001` |
| MinIO / S3 | `http://localhost:9000` |

Experimento ya creado por el ETL: **`HVFHS Driver Pay`** (run `etl_normalize`).
Los nombres internos de Docker (`http://mlflow:5000`, `http://s3:9000`) son para los contenedores, no para Jupyter local.

### Modelo de base (AMq1)

- Algoritmo: **XGBoost** (`XGBRegressor`)
- Params de referencia: `n_estimators=100`, `max_depth=6`, `learning_rate=0.1`
- Métricas en test (escala log): R² ≈ 0.9457, RMSE ≈ 0.1679, MAE ≈ 0.1106

La búsqueda de esta notebook parte de esos valores y prueba combinaciones cercanas.

## 1. Conexión a MinIO y MLflow

`awswrangler` y MLflow hablan con MinIO como si fuera S3. Por eso hay que exportar
credenciales y el endpoint **antes** de leer parquet o loguear artefactos.

- `AWS_ACCESS_KEY_ID` / `AWS_SECRET_ACCESS_KEY`: usuario de MinIO (`minio` / `minio123`)
- `MLFLOW_S3_ENDPOINT_URL` y `AWS_ENDPOINT_URL_S3`: MinIO en `localhost:9000`

Después apuntamos el tracking de MLflow a `localhost:5001` y nos enganchamos al
experimento `HVFHS Driver Pay` (si no existiera, `set_experiment` lo crea).

In [1]:
import awswrangler as wr
import mlflow

# Credenciales y endpoint de MinIO (mismo patrón que test.ipynb / ejemplo de la cátedra)
%env AWS_ACCESS_KEY_ID=minio
%env AWS_SECRET_ACCESS_KEY=minio123
%env MLFLOW_S3_ENDPOINT_URL=http://localhost:9000
%env AWS_ENDPOINT_URL_S3=http://localhost:9000

env: AWS_ACCESS_KEY_ID=minio
env: AWS_SECRET_ACCESS_KEY=minio123
env: MLFLOW_S3_ENDPOINT_URL=http://localhost:9000
env: AWS_ENDPOINT_URL_S3=http://localhost:9000


In [2]:
mlflow_server = "http://localhost:5001"
mlflow.set_tracking_uri(mlflow_server)

experiment_name = "HVFHS Driver Pay"
experiment = mlflow.set_experiment(experiment_name)

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experimento:", experiment.name)
print("Experiment ID:", experiment.experiment_id)

Tracking URI: http://localhost:5001
Experimento: HVFHS Driver Pay
Experiment ID: 1


Si esta celda imprime `HVFHS Driver Pay` y un experiment ID (el del ETL suele ser `1`),
la notebook ya habla con MLflow. En http://localhost:5001, pestaña **Model training**,
debería seguir apareciendo el run `etl_normalize`.

**Siguiente paso:** leer `X_train`, `y_train`, `X_test`, `y_test` desde S3 con `awswrangler`
y verificar formas y columnas. Todavía no entrenamos.

## 2. Lectura de train y test desde MinIO

`awswrangler` usa las variables de entorno de la sección 1 y trata MinIO como S3.
Leemos los cuatro parquet que dejó `normalize_data` (X ya está estandarizado).

El target viene como DataFrame de una columna (`driver_pay_log`). XGBoost espera un
vector 1D, así que después hacemos `.squeeze()`.

No volvemos a escalar ni a partir: eso ya lo hizo el DAG. Si las formas no coinciden
con ~399 817 / ~99 955, paramos y revisamos MinIO.

In [3]:
X_train = wr.s3.read_parquet("s3://data/final/train/hvfhs_X_train.parquet")
y_train = wr.s3.read_parquet("s3://data/final/train/hvfhs_y_train.parquet")
X_test = wr.s3.read_parquet("s3://data/final/test/hvfhs_X_test.parquet")
y_test = wr.s3.read_parquet("s3://data/final/test/hvfhs_y_test.parquet")

y_train = y_train.squeeze()
y_test = y_test.squeeze()

In [4]:
print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_test: ", X_test.shape,  "| y_test: ", y_test.shape)
print("Target:", y_train.name)
print("Features:", list(X_train.columns))
X_train.head()

X_train: (399817, 19) | y_train: (399817,)
X_test:  (99955, 19) | y_test:  (99955,)
Target: driver_pay_log
Features: ['trip_miles_log', 'trip_time_log', 'base_passenger_fare_log', 'tolls', 'bcf', 'sales_tax', 'congestion_surcharge', 'airport_fee', 'tips', 'hora', 'dia_semana', 'es_fin_de_semana', 'franja_horaria', 'es_uber', 'shared_request_flag', 'shared_match_flag', 'access_a_ride_flag', 'wav_request_flag', 'wav_match_flag']


,trip_miles_log,trip_time_log,base_passenger_fare_log,tolls,bcf,sales_tax,congestion_surcharge,airport_fee,tips,hora,dia_semana,es_fin_de_semana,franja_horaria,es_uber,shared_request_flag,shared_match_flag,access_a_ride_flag,wav_request_flag,wav_match_flag
0,0.784228,0.888746,0.245064,-0.304208,-0.057884,0.010520,-0.749093,-0.365643,-0.327022,0.714736,0.828171,0.828171,0.489238,0.599464,-0.181457,-0.131626,-0.033791,-0.052883,-0.320464
1,-0.736098,-0.552508,-0.524975,-0.304208,-0.470383,-0.430777,-0.749093,-0.365643,-0.327022,0.714736,0.828171,0.828171,0.489238,0.599464,-0.181457,-0.131626,-0.033791,-0.052883,-0.320464
2,-1.182582,-1.451329,-1.218863,-0.304208,-0.720829,-0.714830,-0.749093,-0.365643,-0.327022,-1.424804,0.828171,0.828171,-1.592160,0.599464,-0.181457,-0.131626,-0.033791,-0.052883,-0.320464
3,-0.794868,-0.349597,-0.194670,-0.304208,-0.352526,-0.314112,-0.749093,-0.365643,-0.327022,0.714736,-1.207480,-1.207480,0.489238,0.599464,-0.181457,-0.131626,-0.033791,-0.052883,-0.320464
4,0.913283,0.710893,0.573960,-0.304208,0.207294,0.319935,1.346293,-0.365643,-0.327022,1.373056,0.828171,0.828171,1.529937,0.599464,-0.181457,-0.131626,-0.033791,-0.052883,3.120477


Chequeá que:

- Train ≈ **399 817** filas y test ≈ **99 955**
- `y_train` / `y_test` sean 1D y se llamen `driver_pay_log`
- Haya **19 features** (las mismas que armó el ETL: logs, fees, hora, flags, etc.)

Si `read_parquet` falla con error de credenciales o endpoint, volvé a correr las celdas de la sección 1 (el kernel tiene que tener esas variables de entorno).

**Siguiente paso:** búsqueda de hiperparámetros de XGBoost, con un run padre y un run hijo por combinación, logueando R², RMSE y MAE. Todavía no lo escribimos: primero confirmá que los datos se leyeron bien.

## 3. Búsqueda de hiperparámetros (runs anidados)

En MLflow un **experimento** agrupa corridas. Un **run** es un intento concreto (una receta + sus métricas).

Acá usamos **runs anidados**, como el ejemplo de la cátedra:

- **Run padre:** toda la búsqueda (`xgb_grid_search_...`)
- **Run hijo:** cada combinación de hiperparámetros (`nested=True`)

En cada hijo logueamos params (`n_estimators`, `max_depth`, `learning_rate`) y métricas
de **test** en escala log (la del modelo) y en dólares (`np.exp`).

Grilla chica alrededor del XGBoost de AMq1 (`100 / 6 / 0.1`). Son **8 modelos**.
Con ~400 k filas puede tardar **10–20 minutos**; `tree_method="hist"` acelera un poco.

El ganador se elige por **R² de test en log**. Al cerrar el padre, se loguea ese modelo
y se registra en el Model Registry como `hvfhs-driver-pay`.

In [5]:
import datetime
from itertools import product

import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from mlflow.models import infer_signature

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [4, 6],
    "learning_rate": [0.05, 0.1],
}

combinations = list(product(
    param_grid["n_estimators"],
    param_grid["max_depth"],
    param_grid["learning_rate"],
))
print(f"Combinaciones a probar: {len(combinations)}")


def regression_metrics(y_true, y_pred, prefix=""):
    """R², RMSE y MAE. prefix='usd_' para métricas en dólares (después de np.exp)."""
    return {
        f"{prefix}r2": r2_score(y_true, y_pred),
        f"{prefix}rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        f"{prefix}mae": mean_absolute_error(y_true, y_pred),
    }

Combinaciones a probar: 8


In [6]:
run_name_parent = "xgb_grid_search_" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
registered_model_name = "hvfhs-driver-pay"

best = {"r2": -np.inf}

with mlflow.start_run(
    experiment_id=experiment.experiment_id,
    run_name=run_name_parent,
) as parent_run:
    mlflow.set_tags({
        "project": "HVFHS Driver Pay",
        "model_family": "xgboost",
        "search": "grid",
        "target": "driver_pay_log",
    })
    mlflow.log_param("n_combinations", len(combinations))

    for n_estimators, max_depth, learning_rate in combinations:
        child_name = f"xgb_n{n_estimators}_d{max_depth}_lr{learning_rate}"

        with mlflow.start_run(
            experiment_id=experiment.experiment_id,
            run_name=child_name,
            nested=True,
        ):
            model = XGBRegressor(
                n_estimators=n_estimators,
                max_depth=max_depth,
                learning_rate=learning_rate,
                tree_method="hist",
                n_jobs=-1,
                random_state=42,
                verbosity=0,
            )
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

            metrics_log = regression_metrics(y_test, y_pred)
            metrics_usd = regression_metrics(np.exp(y_test), np.exp(y_pred), prefix="usd_")

            mlflow.log_params({
                "n_estimators": n_estimators,
                "max_depth": max_depth,
                "learning_rate": learning_rate,
            })
            mlflow.log_metrics({**metrics_log, **metrics_usd})

            print(
                f"{child_name:28}  "
                f"R2={metrics_log['r2']:.4f}  "
                f"RMSE={metrics_log['rmse']:.4f}  "
                f"MAE={metrics_log['mae']:.4f}"
            )

            if metrics_log["r2"] > best["r2"]:
                best = {
                    "r2": metrics_log["r2"],
                    "rmse": metrics_log["rmse"],
                    "mae": metrics_log["mae"],
                    "usd_r2": metrics_usd["usd_r2"],
                    "usd_rmse": metrics_usd["usd_rmse"],
                    "usd_mae": metrics_usd["usd_mae"],
                    "n_estimators": n_estimators,
                    "max_depth": max_depth,
                    "learning_rate": learning_rate,
                    "model": model,
                }

    # El padre deja constancia de la receta ganadora.
    # El artefacto y el Registry van en la celda siguiente (flavor xgboost).
    mlflow.log_params({
        "best_n_estimators": best["n_estimators"],
        "best_max_depth": best["max_depth"],
        "best_learning_rate": best["learning_rate"],
    })
    mlflow.log_metrics({
        "best_r2": best["r2"],
        "best_rmse": best["rmse"],
        "best_mae": best["mae"],
        "best_usd_r2": best["usd_r2"],
        "best_usd_rmse": best["usd_rmse"],
        "best_usd_mae": best["usd_mae"],
    })

print("Mejor modelo:")
print(
    f"  n_estimators={best['n_estimators']}, "
    f"max_depth={best['max_depth']}, "
    f"learning_rate={best['learning_rate']}"
)
print(f"  test R2 (log) = {best['r2']:.4f}  |  RMSE = {best['rmse']:.4f}  |  MAE = {best['mae']:.4f}")


xgb_n100_d4_lr0.05            R2=0.9786  RMSE=0.1058  MAE=0.0671
xgb_n100_d4_lr0.1             R2=0.9802  RMSE=0.1017  MAE=0.0624
xgb_n100_d6_lr0.05            R2=0.9815  RMSE=0.0983  MAE=0.0585
xgb_n100_d6_lr0.1             R2=0.9823  RMSE=0.0962  MAE=0.0554
xgb_n200_d4_lr0.05            R2=0.9804  RMSE=0.1014  MAE=0.0618
xgb_n200_d4_lr0.1             R2=0.9813  RMSE=0.0989  MAE=0.0589
xgb_n200_d6_lr0.05            R2=0.9824  RMSE=0.0959  MAE=0.0549
xgb_n200_d6_lr0.1             R2=0.9829  RMSE=0.0947  MAE=0.0536

Mejor modelo:
  n_estimators=200, max_depth=6, learning_rate=0.1
  test R2 (log) = 0.9829  |  RMSE = 0.0947  |  MAE = 0.0536


En la UI (**Model training** → **HVFHS Driver Pay**):

- desplegá el padre `xgb_grid_search_...` para ver los 8 hijos
- ordená por `r2` (ganador: `n_estimators=200`, `max_depth=6`, `learning_rate=0.1`, R² ≈ 0.9829)

El registro en el Model Registry lo hace la celda siguiente. No vuelvas a correr la grilla: duplica runs.


## 4. Registrar el mejor modelo

La búsqueda y el registro van en celdas distintas a propósito:

- la celda anterior entrena y loguea métricas (8 runs hijos + receta ganadora en el padre)
- esta celda guarda el artefacto con `mlflow.xgboost.log_model` y lo publica en el Model Registry como `hvfhs-driver-pay`

Hay que usar el flavor **xgboost** (no `mlflow.sklearn.log_model`): XGBRegressor no es un tipo confiable para skops en MLflow 3.

Si `best` sigue en memoria, esta celda no reentrena.


In [ ]:
# Solo registra: no reentrena. Hace falta que `best` siga en memoria (no reinicies el kernel).
assert "model" in best, "No está `best` en memoria. Si reiniciaste el kernel, avisame."

print(
    "Ganador en memoria: "
    f"n_estimators={best['n_estimators']}, "
    f"max_depth={best['max_depth']}, "
    f"learning_rate={best['learning_rate']}, "
    f"R²={best['r2']:.4f}"
)

signature = infer_signature(X_train, best["model"].predict(X_train))

with mlflow.start_run(
    experiment_id=experiment.experiment_id,
    run_name="xgb_register_best",
):
    mlflow.set_tags({"project": "HVFHS Driver Pay", "stage": "register_best"})
    mlflow.log_params({
        "best_n_estimators": best["n_estimators"],
        "best_max_depth": best["max_depth"],
        "best_learning_rate": best["learning_rate"],
    })
    mlflow.log_metrics({
        "best_r2": best["r2"],
        "best_rmse": best["rmse"],
        "best_mae": best["mae"],
        "best_usd_r2": best["usd_r2"],
        "best_usd_rmse": best["usd_rmse"],
        "best_usd_mae": best["usd_mae"],
    })
    mlflow.xgboost.log_model(
        xgb_model=best["model"],
        artifact_path="model",
        signature=signature,
        registered_model_name=registered_model_name,
    )

print(f"Registrado: {registered_model_name}")

Ganador en memoria: n_estimators=200, max_depth=6, learning_rate=0.1, R²=0.9829
